In [5]:
import requests
import json
import time
import sys
from ollama import Client, ResponseError
import os
import base64
from io import BytesIO
from typing import List, Dict, Any

from ollama import chat
from PIL import Image


server_url = "http://192.168.0.18:7860/"
ollama_host = "192.168.0.12"


def get_progress_forever(server_url: str = "http://127.0.0.1:7860"):
    url = f"{server_url}/sdapi/v1/progress"
    while True:
        try:
            # Выполняем GET запрос
            response = requests.get(url, timeout=5)
            
            # Проверяем статус ответа
            if response.status_code == 200:
                data = response.json()
                
                # Доступные ключи в ответе:
                # progress (0.0 - 1.0), eta_relative (время до конца), state (словарь состояния), current_task (название задачи)
                progress_val = data.get('progress', 0.0)
                eta_val = data.get('eta_relative', 0)
                task_name = data.get('current_task', 'Нет активной задачи')
                text_info = data.get('textinfo', '')
                
                # Формируем красивый вывод
                print(f"[{task_name}] Прогресс: {progress_val*100:.1f}% | ETA: {eta_val:.1f}с | Текст: {text_info}")
            else:
                print(f"Ошибка сервера: {response.status_code} - {response.text}")
            time.sleep(1.0)
                
        except requests.exceptions.RequestException as e:
            print(f"Ошибка соединения: {e}. Попытка переподключения...")

#get_progress_forever(server_url)

In [6]:
"""
Скрипт для мониторинга прогресса генерации и памяти через API.
Запускается отдельно от основного приложения.
"""



# Конфигурация
API_URL = server_url
PROGRESS_ENDPOINT = f"{API_URL}/sdapi/v1/progress"
MEMORY_ENDPOINT = f"{API_URL}/sdapi/v1/memory"
TIMEOUT = 5  # Таймаут запроса в секундах


def get_memory_usage():
    """Запрашивает информацию о памяти."""
    try:
        response = requests.get(MEMORY_ENDPOINT, timeout=TIMEOUT)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        return {"error": f"Ошибка запроса памяти: {e}"}


def print_progress(progress_data, memory_data):
    """Печатает информацию о прогрессе и памяти."""
    print("\n" + "="*80)
    print(f"API URL: {API_URL}")
    print(f"Время: {time.strftime('%H:%M:%S')}")
    print("="*80)
    
    # Информация о прогрессе
    if "progress" in progress_data:
        print(f"Прогресс: {progress_data['progress']:.2%}")
    
    if "eta_relative" in progress_data:
        print(f"Осталось (ETA): {progress_data['eta_relative']:.2f} сек")
    
    if "state" in progress_data:
        state = progress_data["state"]
        print(f"Задача: {state.get('job', 'None')}")
        print(f"Шаг: {state.get('sampling_step', 0)}/{state.get('sampling_steps', 0)}")
    
    if "current_task" in progress_data and progress_data["current_task"]:
        print(f"Текущая задача: {progress_data['current_task']}")
    
    if "textinfo" in progress_data:
        # Обрезаем текст если он слишком длинный
        info = str(progress_data["textinfo"])
        if len(info) > 150:
            info = info[:150] + "..."
        print(f"Инфо: {info}")
    
    if "current_image" in progress_data:
        print("Текущее изображение: Есть (Base64)")
    
    print("-" * 40)
    
    # Информация о памяти
    if "ram" in memory_data:
        ram = memory_data["ram"]
        print(f"RAM: {ram.get('used', 0) / 1024 / 1024:.2f} MB / {ram.get('total', 0) / 1024 / 1024:.2f} MB")
    
    if "cuda" in memory_data:
        cuda = memory_data["cuda"]
        if "error" in cuda:
            print(f"CUDA: {cuda['error']}")
        else:
            system = cuda.get("system", {})
            print(f"CUDA: {system.get('used', 0) / 1024 / 1024:.2f} MB / {system.get('total', 0) / 1024 / 1024:.2f} MB")
    
    print("="*80 + "\n")


def monitor():
    """Основной цикл мониторинга."""
    print(f"Запуск монитора для: {API_URL}")
    print("Нажмите Ctrl+C для остановки...")
    
    last_task_id = None
    
    while True:
        try:
            # Параллельно запрашиваем данные
            progress_response = requests.get(PROGRESS_ENDPOINT, timeout=TIMEOUT)
            progress_response.raise_for_status()
            progress_data = progress_response.json()
            
            memory_response = requests.get(MEMORY_ENDPOINT, timeout=TIMEOUT)
            memory_response.raise_for_status()
            memory_data = memory_response.json()
            
            # Определяем текущую задачу для логирования
            current_task = progress_data.get("current_task")
            if current_task and current_task != last_task_id:
                print(f"\n[НОВАЯ ЗАДАЧА] {current_task}")
                last_task_id = current_task
            
            print_progress(progress_data, memory_data)
            
            # Небольшая пауза, чтобы не перегружать сервер
            time.sleep(1)
            
        except requests.exceptions.ConnectionError:
            print("[ОШИБКА] Не удалось подключиться к API. Проверьте, что сервер запущен.")
            time.sleep(5)
        except KeyboardInterrupt:
            print("\nОстановка монитора...")
            break
        except Exception as e:
            print(f"[ОШИБКА] Неожиданная ошибка: {e}")
            time.sleep(5)

#monitor()

In [10]:
def list_models(host: str, port: int = 11434):
    url = f"http://{host}:{port}"
    client = Client(host=url)
    start = time.time()
    try:
        response = client.list()
        elapsed = time.time() - start
        print(f"Request completed in {elapsed:.3f}s")
        
        # Извлекаем список моделей из объекта ListResponse
        models = response.models if hasattr(response, 'models') else []
        
        if models:
            print(f"Found {len(models)} model(s):")
            for i, m in enumerate(models, 1):
                name = m.model if hasattr(m, 'model') else None
                if name:
                    print(f"{i}. {name}")
                else:
                    print(f"{i}. [Unknown model name]")
        else:
            print("No models found.")
    except ResponseError as e:
        elapsed = time.time() - start
        print(f"Request failed in {elapsed:.3f}s: API Error {e.status_code} - {e.error}")
    except ConnectionError:
        elapsed = time.time() - start
        print(f"Request failed in {elapsed:.3f}s: Connection error")
    except Exception as e:
        elapsed = time.time() - start
        print(f"Request failed in {elapsed:.3f}s: {e}")
list_models("192.168.0.12")

Request completed in 0.020s
Found 18 model(s):
1. ticlazau/qwen2.5-coder-7b-instruct:latest
2. qwen3:14b
3. qwen3.5:27b
4. qwen3.5:9b
5. qwen3-embedding:8b
6. qwen3:1.7b
7. qwen3:4b
8. starcoder2:7b
9. qwen2.5-coder:1.5b
10. deepseek-coder:6.7b
11. qwen2.5-coder:14b
12. gemma3:12b
13. mxbai-embed-large:latest
14. llama3.2-vision:latest
15. gemma3:27b
16. deepseek-coder:latest
17. deepseek-r1:latest
18. nomic-embed-text:latest


In [ ]:



class ImageRefinementAgent:
    """
    Агент для итеративной генерации и улучшения изображений.
    """
    def __init__(self, model_name: str = "qwen3-vl:8b"):
        self.model_name = model_name
        # Сохраняем историю диалога (промты, анализ, изображения)
        self.history: List[Dict[str, Any]] = []

    def _encode_image(self, image: Image.Image) -> str:
        """Кодирует PIL Image в строку base64."""
        buffer = BytesIO()
        image.save(buffer, format="PNG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

    def _call_vlm(self, prompt: str, images: List[Image.Image] = None) -> str:
        """
        Внутренний метод для вызова VLM (Qwen3-VL).
        """
        messages = [{'role': 'user', 'content': prompt}]

        if images:
            # Кодируем изображения в base64 для отправки в модель
            encoded_images = [self._encode_image(img) for img in images]
            messages[0]['images'] = encoded_images

        try:
            response = chat(
                model=self.model_name,
                messages=messages,
                # Опционально: для более сложных рассуждений используйте think=True
                # think=True
            )
            return response['message']['content']
        except Exception as e:
            print(f"Ошибка при вызове VLM: {e}")
            return ""

    def _analyze_and_refine(self, original_request: str, generated_image: Image.Image) -> str:
        """
        Анализирует сгенерированное изображение и возвращает улучшенный промт.
        """
        analysis_prompt = f"""
        Твоя задача — действовать как эксперт по генерации изображений. Проанализируй изображение, сгенерированное по запросу: "{original_request}".
        Оцени, насколько хорошо оно соответствует запросу. Выдели сильные стороны и области для улучшения.
        Основываясь на этом анализе, создай **улучшенный промт** для следующей итерации.
        Цель — сделать итоговое изображение максимально качественным и соответствующим исходному запросу.
        Твой ответ должен быть в формате JSON без пояснений:
        {{
        "analysis": "Твой подробный анализ...",
        "improved_prompt": "Улучшенный промт для следующей генерации..."
        }}
        """

        result = self._call_vlm(analysis_prompt, [generated_image])
        return result

    def generate_and_refine(self, initial_prompt: str, max_iterations: int = 3) -> List[Image.Image]:
        """
        Основной метод для генерации и итеративного улучшения изображения.

        Args:
            initial_prompt: Исходный запрос.
            max_iterations: Максимальное количество итераций улучшения.

        Returns:
            Список сгенерированных изображений (включая все итерации).
        """
        current_prompt = initial_prompt
        generated_images = []

        for iteration in range(max_iterations):
            print(f"\n--- Итерация {iteration + 1} ---")
            print(f"Промт: {current_prompt}")

            # 1. Генерируем изображение на основе текущего промта
            # Предполагается, что ваша функция make_image возвращает список PIL изображений
            # new_images = make_image(current_prompt)
            # В этом примере для демонстрации мы используем заглушку
            new_images = self._mock_make_image(current_prompt, iteration)
            if not new_images:
                print("Не удалось сгенерировать изображение.")
                break

            generated_images.extend(new_images)
            # Для простоты используем первое сгенерированное изображение для анализа
            current_image = new_images[0]

            # 2. Если это последняя итерация, выходим из цикла
            if iteration == max_iterations - 1:
                break

            # 3. Анализируем результат и получаем улучшенный промт
            analysis_result = self._analyze_and_refine(initial_prompt, current_image)

            # 4. Извлекаем improved_prompt из JSON-ответа модели
            try:
                import json
                # Пытаемся найти JSON в ответе (на случай, если модель добавила пояснения)
                json_start = analysis_result.find('{')
                json_end = analysis_result.rfind('}') + 1
                if json_start != -1 and json_end != 0:
                    analysis_json = json.loads(analysis_result[json_start:json_end])
                    current_prompt = analysis_json.get("improved_prompt", current_prompt)
                    print(f"Анализ: {analysis_json.get('analysis', 'Нет анализа')}")
                else:
                    print("Не удалось извлечь JSON из ответа модели. Использую предыдущий промт.")
            except json.JSONDecodeError:
                print("Ошибка при разборе JSON. Использую предыдущий промт.")

            print(f"Новый промт: {current_prompt}")

        return generated_images

    def _mock_make_image(self, prompt: str, iteration: int) -> List[Image.Image]:
        """
        Заглушка для функции генерации изображений (make_image).
        В реальном коде здесь будет вызов вашей txt2img модели.
        """
        print(f"  [Заглушка] Генерация изображения по промту: '{prompt}'")
        # Создаем заглушку-изображение (черный квадрат с текстом)
        img = Image.new('RGB', (512, 512), color=(50, 50, 50))
        return [img]